# Notebook 7: Model Evaluation
## Car Price Prediction Project

**Objective:** Deep evaluation of all three models — residual analysis, cross-validation, overfitting check, and business interpretation.

---

## 7.1 Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score

## 7.2 Load Data and Models

In [ ]:
# Load data
X_train_scaled = pd.read_csv('/content/drive/MyDrive/X_train_scaled.csv')
X_test_scaled = pd.read_csv('/content/drive/MyDrive/X_test_scaled.csv')
X_train = pd.read_csv('/content/drive/MyDrive/X_train.csv')
X_test = pd.read_csv('/content/drive/MyDrive/X_test.csv')
y_train = pd.read_csv('/content/drive/MyDrive/y_train.csv').squeeze()
y_test = pd.read_csv('/content/drive/MyDrive/y_test.csv').squeeze()

# Load models
with open('/content/drive/MyDrive/linear_regression_model.pkl', 'rb') as f:
    lr_model = pickle.load(f)
with open('/content/drive/MyDrive/decision_tree_model.pkl', 'rb') as f:
    dt_model = pickle.load(f)
with open('/content/drive/MyDrive/xgboost_model.pkl', 'rb') as f:
    xgb_model = pickle.load(f)

print("Data and models loaded.")

## 7.3 Generate Predictions

In [ ]:
# Predictions
lr_train_pred = lr_model.predict(X_train_scaled)
lr_test_pred = lr_model.predict(X_test_scaled)

dt_train_pred = dt_model.predict(X_train)
dt_test_pred = dt_model.predict(X_test)

xgb_train_pred = xgb_model.predict(X_train)
xgb_test_pred = xgb_model.predict(X_test)

print("Predictions generated for all models.")

## 7.4 Comprehensive Metrics Table

In [ ]:
def full_metrics(name, y_tr, y_tr_pred, y_te, y_te_pred):
    return {
        'Model': name,
        'Train R²': r2_score(y_tr, y_tr_pred),
        'Test R²': r2_score(y_te, y_te_pred),
        'R² Gap': r2_score(y_tr, y_tr_pred) - r2_score(y_te, y_te_pred),
        'Train RMSE': np.sqrt(mean_squared_error(y_tr, y_tr_pred)),
        'Test RMSE': np.sqrt(mean_squared_error(y_te, y_te_pred)),
        'Train MAE': mean_absolute_error(y_tr, y_tr_pred),
        'Test MAE': mean_absolute_error(y_te, y_te_pred),
        'RMSE/Mean Price %': (np.sqrt(mean_squared_error(y_te, y_te_pred)) / y_te.mean()) * 100
    }

metrics = [
    full_metrics('Linear Regression', y_train, lr_train_pred, y_test, lr_test_pred),
    full_metrics('Decision Tree', y_train, dt_train_pred, y_test, dt_test_pred),
    full_metrics('XGBoost', y_train, xgb_train_pred, y_test, xgb_test_pred)
]

metrics_df = pd.DataFrame(metrics).set_index('Model')
metrics_df

## 7.5 Overfitting Analysis

A large gap between Train R² and Test R² indicates overfitting.

In [ ]:
models_names = ['Linear Regression', 'Decision Tree', 'XGBoost']
train_r2 = metrics_df['Train R²'].values
test_r2 = metrics_df['Test R²'].values

x = np.arange(len(models_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, train_r2, width, label='Train R²', color='#3498db')
bars2 = ax.bar(x + width/2, test_r2, width, label='Test R²', color='#e74c3c')

ax.set_ylabel('R² Score')
ax.set_title('Train vs Test R² — Overfitting Check')
ax.set_xticks(x)
ax.set_xticklabels(models_names)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

print("R² Gap (Train - Test):")
for name, gap in zip(models_names, metrics_df['R² Gap'].values):
    status = 'OK' if gap < 0.1 else 'POSSIBLE OVERFIT' if gap < 0.2 else 'OVERFITTING'
    print(f"  {name}: {gap:.4f} → {status}")

## 7.6 Cross-Validation

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

# Full encoded dataset for CV
df_full = pd.read_csv('/content/drive/MyDrive/car_price_encoded.csv')
X_full = df_full.drop(columns=['price'])
y_full = df_full['price']

cv_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=5, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.1,
                             random_state=42, verbosity=0)
}

print("5-Fold Cross-Validation Results (Negative MSE → converted to RMSE):")
print("="*60)
cv_results = []
for name, model in cv_models.items():
    scores = cross_val_score(model, X_full, y_full, cv=5, scoring='neg_mean_squared_error')
    rmse_scores = np.sqrt(-scores)
    cv_results.append({
        'Model': name,
        'CV RMSE Mean': rmse_scores.mean(),
        'CV RMSE Std': rmse_scores.std()
    })
    print(f"  {name}: RMSE = ${rmse_scores.mean():,.0f} ± ${rmse_scores.std():,.0f}")

cv_df = pd.DataFrame(cv_results).set_index('Model')
cv_df

## 7.7 Residual Analysis

In [ ]:
# Residuals for each model
residuals = {
    'Linear Regression': y_test.values - lr_test_pred,
    'Decision Tree': y_test.values - dt_test_pred,
    'XGBoost': y_test.values - xgb_test_pred
}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors = ['#e74c3c', '#2ecc71', '#3498db']
preds = [lr_test_pred, dt_test_pred, xgb_test_pred]

for i, (name, res) in enumerate(residuals.items()):
    # Residual vs Predicted
    axes[0, i].scatter(preds[i], res, alpha=0.6, color=colors[i], s=40)
    axes[0, i].axhline(y=0, color='black', linestyle='--', linewidth=1)
    axes[0, i].set_xlabel('Predicted Price')
    axes[0, i].set_ylabel('Residual')
    axes[0, i].set_title(f'{name} — Residuals vs Predicted')

    # Residual histogram
    axes[1, i].hist(res, bins=20, color=colors[i], edgecolor='white')
    axes[1, i].set_xlabel('Residual')
    axes[1, i].set_ylabel('Frequency')
    axes[1, i].set_title(f'{name} — Residual Distribution')

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7.8 Error Distribution Analysis

In [ ]:
# Percentage error analysis
for name, pred in zip(models_names, preds):
    pct_error = np.abs((y_test.values - pred) / y_test.values) * 100
    print(f"\n{name} — Percentage Error:")
    print(f"  Mean:   {pct_error.mean():.1f}%")
    print(f"  Median: {np.median(pct_error):.1f}%")
    print(f"  <10%:   {(pct_error < 10).sum()} / {len(pct_error)} predictions ({(pct_error < 10).mean()*100:.0f}%)")
    print(f"  <20%:   {(pct_error < 20).sum()} / {len(pct_error)} predictions ({(pct_error < 20).mean()*100:.0f}%)")

## 7.9 RMSE in Context

In [ ]:
print(f"Price Statistics (test set):")
print(f"  Mean:   ${y_test.mean():,.0f}")
print(f"  Median: ${y_test.median():,.0f}")
print(f"  Min:    ${y_test.min():,.0f}")
print(f"  Max:    ${y_test.max():,.0f}")
print(f"  Std:    ${y_test.std():,.0f}")
print(f"  Range:  ${y_test.max() - y_test.min():,.0f}")

print(f"\nRMSE as % of mean price:")
for name in models_names:
    pct = metrics_df.loc[name, 'RMSE/Mean Price %']
    print(f"  {name}: {pct:.1f}%")

print(f"\nInterpretation:")
print(f"  < 10% → Excellent")
print(f"  10-20% → Good")
print(f"  > 20% → Needs improvement")

## 7.10 Best Model Selection

In [ ]:
best_model_name = metrics_df['Test RMSE'].idxmin()
best_rmse = metrics_df.loc[best_model_name, 'Test RMSE']
best_r2 = metrics_df.loc[best_model_name, 'Test R²']

print(f"\n{'='*60}")
print(f"  BEST MODEL: {best_model_name}")
print(f"{'='*60}")
print(f"  Test R²:   {best_r2:.4f}")
print(f"  Test RMSE: ${best_rmse:,.0f}")
print(f"  RMSE/Mean: {metrics_df.loc[best_model_name, 'RMSE/Mean Price %']:.1f}%")
print(f"\n  This model explains {best_r2*100:.1f}% of the variance in price")
print(f"  and its predictions are off by ~${best_rmse:,.0f} on average.")

---
## Evaluation Summary

### R-squared Interpretation
- R² measures the **proportion of variance** in the target variable explained by the model
- R² = 0.91 means 91% of the variation in price is explained by the features
- R² does **NOT** mean accuracy — it does not measure prediction error
- A high R² can still coexist with high RMSE

### RMSE Interpretation
- RMSE measures the **average prediction error** in dollars
- Must be compared to the **range and distribution** of the target variable
- Lower RMSE = better predictive performance

### MAE Interpretation
- MAE gives the average absolute error (treats all errors equally)
- RMSE penalizes large errors more (due to squaring)

**Next step →** Notebook 08: Explainability